# continuum-lab: calibrate and grade

Fit profile distributions from chain Parquet tables (mithril2pq), then grade a devnet run against them. See `README.md` for what each statistic means.

New to the statistics? The [guide notebook](guide.ipynb) walks through the ideas with sliders and the real preprod and preview fits.

In [ ]:
import sys
sys.path.insert(0, '/work/lab')
import common, fit, grade
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CHAIN_DIR = '/data/chain'      # mithril2pq output (mainnet, preview, ...)
EXPORT_DIR = '/data/export'    # `continuum export` output of a run
DEVNET_CHAIN_DIR = None        # optional: mithril2pq tables of the devnet itself

## Fit

In [ ]:
chain = common.chain_stats(CHAIN_DIR)
yaml_text, target = fit.build(chain, 'chain-fit')
print(yaml_text)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].hist(chain.samples['log10_output_value'], bins=80); ax[0].set_title('log10 output value (lovelace)')
ax[1].hist(chain.samples['log10_gap_seconds'], bins=80); ax[1].set_title('log10 inter-tx gap (s)')
ax[2].hist(np.log10(chain.samples['activity_per_day']), bins=60); ax[2].set_title('log10 txs per address per day')
plt.tight_layout()

## Grade

In [ ]:
run = common.engine_stats(EXPORT_DIR)
if DEVNET_CHAIN_DIR:
    run = common.merge_chain(run, common.chain_stats(DEVNET_CHAIN_DIR))
rows = grade.grade(run, target, dmax=0.1, tvmax=0.1, fmax=0.1)
pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, k in zip(ax, ['log10_output_value', 'log10_gap_seconds']):
    a.hist(target['quantiles'][k], bins=60, alpha=.5, density=True, label='target')
    a.hist(run.samples[k], bins=60, alpha=.5, density=True, label='run'); a.set_title(k); a.legend()
plt.tight_layout()